# Тема 6 — Где колено насыщения? Инцидент, который вы расследуете

**Что случилось.** Сервис классификации тикетов поддержки отлично работал на 8
пользователей: ответы приходили за доли секунды, всё в дедлайне. В час пик пришло 128 —
и сервис начал возвращать таймауты, клиенты повторяли запросы, очередь росла быстрее,
чем разгружалась. Модель та же, GPU тот же, код тот же. **Что кончилось первым?**

**Чем занимается сервис.** Приложение **A** читает тикеты поддержки («My subscription
renewed once, but two settled charges appeared. Please refund the duplicate payment.»)
и выбирает одну категорию: `billing`, `bug`, `access`… Приложение **B** (stretch) делает
то же для длинных документов — например, разбора, оплачен ли принятый work за тридцать
дней. Запрос — это текст; ответ — короткая метка, два-три токена. Ниже в первой же клетке
вы увидите живые тексты корпуса и распределение их длин — это ваша нагрузка, посмотрите
на неё глазами до любых формул.

**Ваша лаба, по шагам.** (1) Посмотреть на данные, которыми живёт сервис. (2) Посчитать
бюджет памяти и **заморозить прогноз** колена до измерений. (3) Прогнать волны 8/32/64/128,
построить водопад запросов и три графика. (4) Найти колено по объявленному правилу,
выбрать рабочую точку под SLO и защитить её письменно. (5) Stretch: второе приложение
делит сервер — измерить интерференцию и защитить in-flight cap.

Профили: **mock** (по умолчанию, бесплатный CPU, минуты) — объявленная синтетическая
модель очереди для отладки конвейера; **gpu** (инструктор) — настоящий vLLM на T4.
Никакие числа из mock не являются измерениями реального сервиса — это сказано на каждом
графике. Конверт: 2 контактных часа + 3–5 самостоятельных.

**Правила игры.** (1) Ошибки — данные: таймаут пишется в журнал, а не
выбрасывается; ретраев нет. (2) Прогноз фиксируется файлом **до** измерений — и
в конце сверяется с фактом; предсказание после прогона — не прогноз, а история.
(3) Каждый график читается вместе со своим знаменателем: сколько попыток в него
вошло. (4) Семь функций с контрактами в docstring — это и есть задание; остальное —
предоставленная инфраструктура (свёрнута и молчит). Если клетка с вашим TODO падает
с NotImplementedError — так и задумано: реализуйте контракт и запускайте самопроверку
следом в той же клетке.

In [ ]:
import json, os, time, threading, hashlib, urllib.request
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from typing import TypedDict, Sequence

PROFILE = os.environ.get('LSM_T06_PROFILE', 'mock')  # 'mock' | 'gpu' (owner-run only)
ATTEMPT = 'attempt-001'
RESULTS = os.path.join('results', PROFILE, ATTEMPT)
os.makedirs(RESULTS, exist_ok=True)
print(f'profile={PROFILE}  results dir={RESULTS}')

In [ ]:
class ModelConfig(TypedDict):
    """Serving-relevant model facts, declared once and shared by every calculation."""
    n_layers: int; n_kv_heads: int; head_dim: int; dtype_bytes: int

MODEL_FACTS: ModelConfig = dict(n_layers=36, n_kv_heads=2, head_dim=128, dtype_bytes=2)

class RequestRecord(TypedDict):
    """One client-observed request. Timestamps are client-side monotonic seconds."""
    request_id: str; wave: int; client_id: int; app_id: str; case_id: str
    input_tokens: int; max_tokens: int
    arrival_ts: float; first_byte_ts: float | None; terminal_ts: float
    output_tokens: int; http_status: int | None; outcome: str

class Percentiles(TypedDict):
    """Nearest-rank TTFT percentiles for one wave, completed requests only."""
    wave: int; count: int; p50_ms: float; p95_ms: float

class AuditSummary(TypedDict):
    """Damage classification of one wave's journal; nothing is repaired."""
    records: int; unique_ids: int; missing_terminal: int; duplicate_ids: int
    foreign_waves: int; impossible_time: int; complete: int

class Decision(TypedDict):
    """The chosen operating wave with its evidence and named binding constraint."""
    wave: int; throughput_tok_per_s: float; ttft_p95_ms: float; error_rate: float
    satisfies_slo: bool; reason: str

class InterferenceReport(TypedDict):
    """A-app degradation when B shares the server, plus B's completed count."""
    a_throughput_ratio: float; a_ttft_p95_ratio: float
    a_completed_alone: int; a_completed_shared: int; b_completed_shared: int

def save_jsonl(path: str, rows: Sequence[dict]) -> str:
    """Write a JSONL journal and return its sha256 hex digest.

    Parameters
    ----------
    path : str
        Destination file path, overwritten.
    rows : sequence of dict
        Records to serialize, one JSON object per line.

    Returns
    -------
    str
        sha256 of the written bytes, for provenance receipts.
    """
    with open(path, 'w', encoding='utf-8') as fh:
        for row in rows:
            fh.write(json.dumps(row, ensure_ascii=False) + chr(10))
    return hashlib.sha256(Path_bytes(path)).hexdigest()

def Path_bytes(path: str) -> bytes:
    """Read a file as raw bytes.

    Parameters
    ----------
    path : str
        File to read.

    Returns
    -------
    bytes
        Exact file content, for hashing.
    """
    with open(path, 'rb') as fh:
        return fh.read()

def load_jsonl(path: str) -> list[dict]:
    """Load a JSONL file into a list of dicts.

    Parameters
    ----------
    path : str
        Journal file, one JSON object per line.

    Returns
    -------
    list of dict
        Parsed records, in file order.
    """
    with open(path, encoding='utf-8') as fh:
        return [json.loads(line) for line in fh if line.strip()]

In [ ]:
# Данные, которыми занимается сервис: живые тексты замороженного корпуса.
import pandas as pd
from IPython.display import display
PINNED_CORPUS_URL = (
    'https://raw.githubusercontent.com/levshaazz/'
    'llm-serving-mastery-submission-example/'
    '8525bd29959562db69800bf453555155f2ade668/classroom/topic06-purpose-v4/corpus.jsonl')
if not os.path.exists('corpus.jsonl'):
    urllib.request.urlretrieve(PINNED_CORPUS_URL, 'corpus.jsonl')
corpus_rows = [json.loads(l) for l in open('corpus.jsonl', encoding='utf-8') if l.strip()]
sample = ([r for r in corpus_rows if r['app_id'] == 'A'][:3]
          + [r for r in corpus_rows if r['app_id'] == 'B'][:1])
_tbl = pd.DataFrame([{'app': r['app_id'], 'case_id': r['case_id'], 'gold': r['gold'],
                      'tokens': r['input_tokens'],
                      'текст (начало)': r['user_text'][:80] + '…'} for r in sample])
display(_tbl.style.set_caption(
    f'всего {len(corpus_rows)} текстов замороженного корпуса;'
    ' A-тикет ≈ 100 токенов, B-документ ≈ 500; gold — правильная метка, не обещание модели'))

In [ ]:
# Та же нагрузка глазами длин: где живут A и B.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3.2))
for app, color in (('A', '#2563eb'), ('B', '#d97706')):
    lens = [r['input_tokens'] for r in corpus_rows if r['app_id'] == app]
    ax.hist(lens, bins=30, alpha=0.55, label=f'приложение {app} (n={len(lens)})', color=color)
ax.set_xlabel('длина входа, токены')
ax.set_ylabel('число текстов')
ax.set_title('Замороженный корпус: A-тикет против B-документа')
ax.legend(); fig.tight_layout(); plt.show()

**Что вы должны увидеть в данных.** A-тикет — это
одно-два предложения человеческого текста с конкретной жалобой; B-документ — в пять
раз длиннее, с процедурным хвостом. Метка `gold` — правильный ответ человека, а не
обещание модели: она может ошибаться, и в Topic 04 вы видели, как именно. Для нагрузки
это значит: A-волна состоит из коротких префиллов (~100 токенов) и почти мгновенных
ответов-меток; B тянет за собой длинный префилл. Когда в stretch оба приложения
сойдутся на одном сервере, разница длин станет источником интерференции: чей префилл
займёт следующий шаг шедулера — тот и решит, кто подождёт.

## Блок 0 — Прогноз до измерений

У инцидента «что кончилось первым» два главных подозреваемых: **compute** (префилл нового
запроса — полный проход по всем весам, конкурирующий со всем батчем) и **KV-память**
(каждый удержанный токен стоит байты в каждом слое). Второго подозреваемого можно
посчитать на салфетке — и вы сделаете это раньше, чем подадите первую нагрузку.

<img src="https://levshaazz.github.io/llm-serving-mastery/Lectures/assets/figures/t6-kv-budget-light.svg" width="860"/>

Модель та же, что во всей теме: Qwen2.5-3B, FP16, 36 слоёв, 2 KV-головы, ширина головы
128. Заметьте, насколько это дёшево: 16 query-голов делят всего 2 KV-головы (GQA) — если
по ошибке подставить query-головы, счёт завысится в восемь раз. На T4 15 360 МиБ, из них
≈ 6350 МиБ — веса. **Сколько одновременных полных контекстов влезает в остаток?**
Посчитайте функцией (множитель 2 в начале — это K и V, два тензора), покажите рост
столбиками, запишите прогноз колена — файл хэшируется до любого измерения, и в конце вы
сведёте прогноз с фактом. Арифметический потолок — граница сверху, не обещание шедулера:
фрагментация, активации и политика резервирования съедают часть, и в мемо вам предстоит
назвать, кто именно.

In [ ]:
def kv_budget_bytes(cfg: ModelConfig, context_tokens: int) -> int:
    """Total KV-cache bytes one request occupies for a fully retained context.

    Parameters
    ----------
    cfg : ModelConfig
        Declared model facts: layers, KV heads, head width, bytes per element.
    context_tokens : int
        Retained positions for one request, >= 0.

    Returns
    -------
    int
        Bytes for K and V together across all layers for this one request.

    Raises
    ------
    ValueError
        If ``context_tokens`` is negative or any cfg field is not a positive int.

    Notes
    -----
    Arithmetic on declared facts. It is an upper-bound ceiling, not vLLM's
    allocation policy: fragmentation, activations and swap are not included.
    """
    raise NotImplementedError('TODO: derive the per-position and per-request byte cost')

# --- самопроверка (упадёт, пока функция не реализована) ---
assert kv_budget_bytes(MODEL_FACTS, 1) == 36_864
assert kv_budget_bytes(MODEL_FACTS, 0) == 0
try:
    kv_budget_bytes(MODEL_FACTS, -3); raise SystemExit('negative context must raise')
except ValueError:
    pass
print('kv_budget_bytes: 36 864 байта на позицию — сходится с арифметикой на слайде')

In [ ]:
# Рост цены запроса глазами. Меняйте длины и смотрите, как растут столбцы.
contexts = [512, 1024, 2048]
costs_mib = [kv_budget_bytes(MODEL_FACTS, c) / 2**20 for c in contexts]
fig, ax = plt.subplots(figsize=(5.2, 3))
bars = ax.bar([str(c) + ' ток.' for c in contexts], costs_mib, color='#2563eb')
for b, m in zip(bars, costs_mib):
    ax.annotate(f'{m:.0f} МиБ', (b.get_x() + b.get_width() / 2, b.get_height()),
                ha='center', va='bottom')
ax.set_ylabel('МиБ на запрос')
ax.set_title('KV-бюджет одного запроса (2048-й контекст ≈ 72 МиБ)')
fig.tight_layout(); plt.show()

In [ ]:
T4_USABLE_BYTES = 15_360 * 1024 * 1024
WEIGHT_BYTES = int(6.2 * 1024**3)
CTX_TOKENS = 2048
per_request = kv_budget_bytes(MODEL_FACTS, CTX_TOKENS)
ceiling = (T4_USABLE_BYTES - WEIGHT_BYTES) // per_request
predicted_knee = 32   # TODO: ваш прогноз — наибольшая волна, которая останется здоровой
prediction_rationale = 'TODO: 1–2 предложения: какой ресурс переварит первым и почему'
pred = dict(per_request_bytes=per_request, ceiling_requests=ceiling,
            predicted_knee_wave=predicted_knee, rationale=prediction_rationale,
            profile=PROFILE, frozen_at=time.time())
with open(os.path.join(RESULTS, 'predictions.json'), 'w') as fh:
    json.dump(pred, fh, ensure_ascii=False, indent=1)
print(f'потолок (арифметика) = {ceiling} запросов при {CTX_TOKENS} токенах; прогноз заморожен')

## Блок 1 — Измерение: волны, журнал, водопад

**Дизайн эксперимента.** Меняется ровно один фактор — конкурентность. Почему волны, а
не «запросов в секунду»? Потому что req/s у замкнутого цикла — не вход, а следствие:
8 клиентов с ответом за 0.25 с дают ≈32 req/s, замедли сервис вдвое — и спрос упадёт
вдвое сам (закон Литтла в замкнутой форме). Конкурентность — единственная руль, которую
вы держите честно. Волны 8/32/64/128 закрытых клиентов; каждый шлёт следующий запрос
только после завершения предыдущего.
Зафиксировано: модель, корпус, лимит вывода, температура, бюджет на запрос. Драйвер
**никогда не повторяет** запрос: ретрай молча удвоил бы спрос и спрятал отказ. Каждый
завершённый шанс пишет одну запись журнала с клиентскими часами: приход → первый байт →
терминал, плюс исход.

Пять исходов — данные, а не шум: `completed`, `timeout` (это **клиент** сдался, а не сервер
сломался), `server_error`, `connection_error`, `truncated`. Часы у каждой записи —
клиентские и монотонные: приход → первый байт → терминал. Время до первого токена (TTFT)
и время на токен после него (TPOT) — разная физика: первое — очередь плюс префилл, второе —
интерференция декода. Под нагрузкой первыми растут именно они, и по тому, какое удвоилось,
ставится диагноз.

После волн вы увидите **сводную таблицу** по волнам и **водопад запросов**: горизонтальная
полоса на строку — серый зазор (ожидание до первого токена), синее (обслуживание), красное
(не дождался). Это главная картинка темы: **ожидание — не обслуживание**. Прежде чем
верить таблице — аудит: пропавшие терминалы, дубли ID, чужие волны и невозможные времена
отменяют доверие журналу целиком.

In [ ]:
class MockQueueServer:
    """Declared synthetic contention model. Validation plumbing, NOT a measurement.

    Per request: admission queue delay 0.9 * in_flight**2 ms, prefill 1.2 ms/token,
    decode 8 ms/token * (1 + 0.06 * (in_flight - 1)). Deterministic given the
    in-flight count observed at admission.
    """
    def __init__(self) -> None:
        """Create a server with zero requests in flight.

        Returns
        -------
        None
            Stateful helper; see request().
        """
        self._lock = threading.Lock()
        self._in_flight = 0

    def request(self, case: dict, max_tokens: int) -> tuple[str, int]:
        """Serve one synthetic request.

        Parameters
        ----------
        case : dict
            Load case with input_tokens.
        max_tokens : int
            Requested output tokens.

        Returns
        -------
        tuple of (str, int)
            Output text and output token count.
        """
        input_tokens = case['input_tokens']
        with self._lock:
            n = self._in_flight = self._in_flight + 1
        try:
            queue_ms = 0.9 * n * n
            ttft = (0.15 + queue_ms / 1000 + 1.2 * input_tokens / 1000)
            per_tok = 0.008 * (1 + 0.06 * (n - 1))
            time.sleep(ttft)
            time.sleep(per_tok * max_tokens)
            return 'SYNTHETIC-MOCK-OUTPUT', max_tokens
        finally:
            with self._lock:
                self._in_flight -= 1

def run_wave(runner: object, app_id: str, cases: Sequence[dict], wave: int, per_client: int,
             timeout_s: float) -> list[RequestRecord]:
    """Run one closed-loop wave of `wave` clients, `per_client` requests each.

    Parameters
    ----------
    runner : object
        Service with .request(case, max_tokens); mock here, HTTP on gpu.
    app_id : str
        Application label written into every record.
    cases : sequence of dict
        Load cases with case_id and input_tokens (mock) or rendered (gpu).
    wave : int
        Concurrency: number of simultaneous closed-loop clients.
    per_client : int
        Sequential requests each client sends.
    timeout_s : float
        Per-request wall budget before the client gives up.

    Returns
    -------
    list of RequestRecord
        One record per terminated attempt, outcomes classified.
    """
    records: list[RequestRecord] = []
    lock = threading.Lock()

    def client(cid: int) -> None:
        """Send this client's sequential requests and journal each attempt.

        Parameters
        ----------
        cid : int
            Client index inside the wave.

        Returns
        -------
        None
            Results are appended to the shared records list.
        """
        for seq in range(per_client):
            case = cases[(cid * per_client + seq) % len(cases)]
            rid = f'w{wave}-c{cid}-r{seq}'
            t0 = time.monotonic()
            outcome, status, out_tok, fb = 'connection_error', None, 0, None
            try:
                text, out_tok = runner.request(case, 32)
                fb = time.monotonic()
                outcome, status = ('completed', 200) if out_tok > 0 else ('truncated', 200)
                if time.monotonic() - t0 > timeout_s:
                    outcome = 'timeout'
            except TimeoutError:
                outcome = 'timeout'
            except Exception as exc:  # noqa: BLE001 - transport failures are data
                outcome = 'server_error' if '5xx' in str(exc) else 'connection_error'
            rec = RequestRecord(request_id=rid, wave=wave, client_id=cid, app_id=app_id,
                                case_id=case['case_id'], input_tokens=case['input_tokens'],
                                max_tokens=32, arrival_ts=t0, first_byte_ts=fb,
                                terminal_ts=time.monotonic(), output_tokens=out_tok,
                                http_status=status, outcome=outcome)
            if time.monotonic() - t0 > timeout_s and outcome == 'completed':
                rec['outcome'] = 'timeout'
            with lock:
                records.append(rec)

    with ThreadPoolExecutor(max_workers=wave) as pool:
        list(pool.map(client, range(wave)))
    return records

def run_shared_wave(server: object, b_cap_k: int, a_cases: Sequence[dict], b_cases: Sequence[dict],
                    n_a: int, n_b: int, per_client: int, timeout_s: float) -> list[RequestRecord]:
    """One concurrent wave: n_a A-clients plus n_b B-clients capped at b_cap_k.

    Parameters
    ----------
    server : object
        Runner shared by both applications; overlap IS the interference.
    b_cap_k : int
        In-flight cap for B, one of 1/2/4/8/16.
    a_cases : sequence of dict
        Load cases for application A.
    b_cases : sequence of dict
        Load cases for application B.
    n_a : int
        A client count.
    n_b : int
        B client count.
    per_client : int
        Sequential requests per client.
    timeout_s : float
        Per-request wall budget.

    Returns
    -------
    list of RequestRecord
        Merged A+B journal; app_id and the B- request-id prefix distinguish them.
    """
    capped = InFlightCapRunner(server, b_cap_k)
    with ThreadPoolExecutor(max_workers=n_a + n_b) as pool:
        fut_a = pool.submit(run_wave, server, 'A', a_cases, n_a, per_client, timeout_s)
        fut_b = pool.submit(run_wave, capped, 'B', b_cases, n_b, per_client, timeout_s)
        a_recs, b_recs = fut_a.result(), fut_b.result()
    for r in b_recs:
        r['request_id'] = 'B-' + r['request_id']
    return a_recs + b_recs

In [ ]:
def _wave_stats(records: Sequence[RequestRecord], w: int) -> dict:
    """Compute per-wave summary statistics for the table.

    Parameters
    ----------
    records : sequence of RequestRecord
        Full journal.
    w : int
        Wave label.

    Returns
    -------
    dict
        n, completed, timeouts, wall_s, throughput, p50_ms, p95_ms.
    """
    rs = [r for r in records if r['wave'] == w]
    done = [r for r in rs if r['outcome'] == 'completed']
    wall = max(r['terminal_ts'] for r in rs) - min(r['arrival_ts'] for r in rs)
    tt = sorted((r['first_byte_ts'] - r['arrival_ts']) * 1000
                for r in done if r['first_byte_ts'])
    return dict(wave=w, n=len(rs), completed=len(done),
                timeouts=sum(1 for r in rs if r['outcome'] == 'timeout'),
                wall_s=round(wall, 2),
                throughput=round(sum(r['output_tokens'] for r in done) / wall, 1),
                p50_ms=round(tt[len(tt) // 2]) if tt else None,
                p95_ms=round(tt[-(-95 * len(tt) // 100) - 1]) if tt else None)

def summary_rows(records: Sequence[RequestRecord], waves: Sequence[int]) -> 'pd.DataFrame':
    """Build the per-wave summary table as a DataFrame indexed by wave.

    Parameters
    ----------
    records : sequence of RequestRecord
        Full journal.
    waves : sequence of int
        Wave labels to include.

    Returns
    -------
    pandas.DataFrame
        One row per wave with outcomes and timing columns.
    """
    import pandas as pd
    return pd.DataFrame([_wave_stats(records, w) for w in waves]).set_index('wave')

def plot_curve(xs: Sequence[int], ys: Sequence[float], ylabel: str, path: str,
               title: str = '', slo_ms: float | None = None,
               knee: int | None = None) -> None:
    """One annotated curve: value labels, optional SLO line and knee marker.

    Parameters
    ----------
    xs : sequence of int
        Concurrency values.
    ys : sequence of float
        Metric values, same length as xs.
    ylabel : str
        Y-axis label with units.
    path : str
        File name inside RESULTS.
    title : str, default ''
        Optional figure title.
    slo_ms : float or None, default None
        SLO reference line in milliseconds.
    knee : int or None, default None
        Knee wave to annotate with an arrow.

    Returns
    -------
    None
        Writes the PNG into RESULTS and shows it.
    """
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.plot(xs, ys, marker='o', color='#2563eb')
    for x, y in zip(xs, ys):
        ax.annotate(f'{y:.4g}', (x, y), textcoords='offset points', xytext=(0, 8),
                    ha='center', fontsize=9)
    if slo_ms is not None:
        ax.axhline(slo_ms, color='#dc2626', linestyle='--', lw=1.4)
        ax.annotate(f'SLO {slo_ms:g}', (0.01, slo_ms), xycoords=('axes fraction', 'data'),
                    color='#dc2626', va='bottom', fontsize=9)
    if knee is not None and knee in list(xs):
        yk = ys[list(xs).index(knee)]
        ax.annotate('колено', (knee, yk), textcoords='offset points', xytext=(12, -18),
                    color='#15803d', fontsize=10,
                    arrowprops=dict(arrowstyle='->', color='#15803d'))
    ax.set_xlabel('конкурентность (закрытый цикл)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS, path), dpi=120, bbox_inches='tight')
    plt.show()

def outcome_shares(records: Sequence[RequestRecord], waves: Sequence[int], path: str) -> None:
    """Stacked per-wave outcome shares with percent labels; all records counted.

    Parameters
    ----------
    records : sequence of RequestRecord
        Full journal.
    waves : sequence of int
        Wave labels to plot.
    path : str
        File name inside RESULTS.

    Returns
    -------
    None
        Writes the PNG into RESULTS and shows it.
    """
    kinds = ['completed', 'timeout', 'server_error', 'connection_error', 'truncated']
    fig, ax = plt.subplots(figsize=(6.5, 4))
    bottoms = [0.0] * len(waves)
    for kind in kinds:
        shares = []
        for w in waves:
            rs = [r for r in records if r['wave'] == w]
            shares.append(sum(1 for r in rs if r['outcome'] == kind) / len(rs) if rs else 0.0)
        ax.bar([str(w) for w in waves], shares, bottom=bottoms, label=kind)
        for i, (b, s) in enumerate(zip(bottoms, shares)):
            if s >= 0.06:
                ax.annotate(f'{s * 100:.0f}%', (i, b + s / 2), ha='center',
                            va='center', fontsize=8, color='white')
        bottoms = [b + s for b, s in zip(bottoms, shares)]
    ax.set_xlabel('конкурентность'); ax.set_ylabel('доля завершённых попыток')
    ax.set_title('Исходы по волнам (в знаменателе — каждая попытка)')
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS, path), dpi=120, bbox_inches='tight')
    plt.show()

def plot_waterfall(records: Sequence[RequestRecord], wave: int, path: str,
                   max_rows: int = 40) -> None:
    """Request waterfall: grey wait, blue service, red timeout, one bar per request.

    Parameters
    ----------
    records : sequence of RequestRecord
        Full journal.
    wave : int
        Wave label to draw.
    path : str
        File name inside RESULTS.
    max_rows : int, default 40
        Cap on drawn requests, earliest arrivals first.

    Returns
    -------
    None
        Writes the PNG into RESULTS and shows it.
    """
    rs = sorted((r for r in records if r['wave'] == wave),
                key=lambda r: r['arrival_ts'])[:max_rows]
    t0 = min(r['arrival_ts'] for r in rs)
    fig, ax = plt.subplots(figsize=(8.5, max(3.0, 0.22 * len(rs))))
    for i, r in enumerate(rs):
        a = r['arrival_ts'] - t0
        if r['outcome'] == 'timeout' or r['first_byte_ts'] is None:
            ax.barh(i, r['terminal_ts'] - t0 - a, left=a, height=0.7, color='#dc2626')
        else:
            fb = r['first_byte_ts'] - t0
            ax.barh(i, fb - a, left=a, height=0.7, color='#9ca3af')
            ax.barh(i, r['terminal_ts'] - t0 - fb, left=fb, height=0.7, color='#2563eb')
    ax.set_yticks([])
    ax.set_xlabel('секунды от начала волны')
    ax.set_title(f'Водопад запросов, волна {wave} (первые {len(rs)})')
    import matplotlib.patches as mpatches
    ax.legend(handles=[mpatches.Patch(color='#9ca3af', label='ожидание'),
                       mpatches.Patch(color='#2563eb', label='обслуживание'),
                       mpatches.Patch(color='#dc2626', label='таймаут')], fontsize=8)
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS, path), dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# GPU PROFILE ONLY. Pinned vLLM install + owned child service in one bootstrap.
if PROFILE != 'gpu':
    print('skipped: gpu-only cell (profile is mock)')
else:
    import subprocess  # noqa: E402
    out = subprocess.run(['pip', 'install', '-q', 'vllm==0.30.0'],
                         capture_output=True, text=True)
    assert out.returncode == 0, ('vLLM install failed: ' + (out.stderr or '')[-300:])
    SERVER_CMD = ['python', '-m', 'vllm.entrypoints.openai.api_server',
                  '--model', 'Qwen/Qwen2.5-3B-Instruct',
                  '--revision', 'aa8e72537993ba99e69dfaafa59ed015b17504d1',
                  '--served-model-name', 'submission', '--dtype', 'float16',
                  '--max-model-len', '2048', '--port', '8017']
    server = subprocess.Popen(SERVER_CMD,
                              stdout=open(os.path.join(RESULTS, 'server.log'), 'w'),
                              stderr=subprocess.STDOUT)
    print('server pid', server.pid, '- the next cell waits for /health')

In [ ]:
# GPU PROFILE ONLY. Bounded readiness wait; never wave into a dead server.
if PROFILE != 'gpu':
    print('skipped: gpu-only cell (profile is mock)')
else:
    import urllib.error  # noqa: E402
    waited = 0
    healthy = False
    deadline = time.monotonic() + 900
    while not healthy:
        try:
            with urllib.request.urlopen('http://127.0.0.1:8017/health', timeout=5) as r:
                healthy = r.status == 200
        except (urllib.error.URLError, OSError):
            pass
        assert time.monotonic() < deadline, 'service did not become healthy in 900 s'
        if not healthy:
            time.sleep(5); waited += 5
    print(f'service healthy; waited about {waited} s')

In [ ]:
# GPU PROFILE ONLY. Frozen public corpus (pinned example revision), D/V splits.
if PROFILE != 'gpu':
    print('skipped: gpu-only cell (profile is mock)')
else:
    def _cases(app: str, split: str) -> list[dict]:
        """Select load cases for one application and split.

        Parameters
        ----------
        app : str
            Application label, 'A' or 'B'.
        split : str
            Corpus split, 'D' (development) or 'V' (held-out).

        Returns
        -------
        list of dict
            Cases with case_id, input_tokens and rendered prompt.
        """
        return [dict(case_id=r['case_id'], input_tokens=r['input_tokens'],
                     rendered=r['rendered_text'])
                for r in corpus_rows if r['split'] == split and r['app_id'] == app
                and r['length_bin'] == 'short']
    gpu_cases_A, gpu_cases_B = _cases('A', 'D'), _cases('B', 'D')
    heldout_A, heldout_B = _cases('A', 'V'), _cases('B', 'V')
    print(f'dev A={len(gpu_cases_A)} B={len(gpu_cases_B)}  '
          f'heldout A={len(heldout_A)} B={len(heldout_B)}')

In [ ]:
class HttpRunner:
    """Synchronous completions client for the local pinned service (case dict in)."""
    def __init__(self, port: int = 8017) -> None:
        """Point the client at one local service port.

        Parameters
        ----------
        port : int, default 8017
            Port of the owned vLLM service.

        Returns
        -------
        None
            Connectionless helper; see request().
        """
        self.url = f'http://127.0.0.1:{port}/v1/completions'

    def request(self, case: dict, max_tokens: int) -> tuple[str, int]:
        """Complete one rendered prompt.

        Parameters
        ----------
        case : dict
            Load case with the rendered prompt under 'rendered'.
        max_tokens : int
            Output token cap.

        Returns
        -------
        tuple of (str, int)
            Output text and completion token count.
        """
        body = json.dumps(dict(model='submission', prompt=case['rendered'],
                               max_tokens=max_tokens, temperature=0.0)).encode()
        req = urllib.request.Request(self.url, data=body,
                                     headers={'Content-Type': 'application/json'})
        with urllib.request.urlopen(req, timeout=120) as resp:
            data = json.loads(resp.read())
        return data['choices'][0]['text'], data['usage']['completion_tokens']

In [ ]:
WAVES = [8, 32, 64, 128]
if PROFILE == 'gpu':
    RUNNER, CASES, TMO = HttpRunner(), gpu_cases_A, 60.0
else:
    MOCK_CASES = [dict(case_id=f'synth-{i}', input_tokens=n)
                  for i, n in enumerate([100, 110, 120] * 40)]
    RUNNER, CASES, TMO = MockQueueServer(), MOCK_CASES, 12.0
all_records: list[RequestRecord] = []
for w in WAVES:
    recs = run_wave(RUNNER, 'A', CASES, w, per_client=3, timeout_s=TMO)
    all_records.extend(recs)
print('волны завершены:', {w: sum(1 for r in all_records if r['wave'] == w) for w in WAVES},
      'записей; дальше — сводная таблица и водопад')

In [ ]:
# Сводная таблица волн: исходы и производительность рядом; журнал сохраняется молча.
journal_path = os.path.join(RESULTS, 'raw-journal.jsonl')
journal_sha = save_jsonl(journal_path, all_records)
display(summary_rows(all_records, WAVES).style.set_caption(
    f'raw-journal.jsonl сохранён ({len(all_records)} записей, sha {journal_sha[:10]}…)'))

In [ ]:
# Водопад запросов волны 128: серый зазор = ожидание, синее = обслуживание, красное = таймаут.
plot_waterfall(all_records, wave=128, path='waterfall.png')

**Что вы должны увидеть** (mock-профиль): на волне 128 у большинства запросов — длинный
серый зазор ожидания и **красная полоса-таймаут**: клиент сдался раньше, чем дошло до
обслуживания; у трети запросов, пробившихся раньше всех, синее обслуживание короткое.
Поменяйте аргумент
на 8 и перезапустите клетку — зазоры исчезнут, всё синее.

**Вопрос себе:** где провёл время запрос, который не получил ответа — в обслуживании
или в очереди? Это «ожидание ≠ обслуживание» и есть механизм колена: лишние клиенты
покупают место в очереди, а не токены.

Попробуйте также волну 32: красного почти нет, но серые зазоры уже неровные — это
интерференция префиллов: чужой префилл забирает шаг, и ваш токен опаздывает, хотя
никакой ошибки не произошло. Сравните три водопада (8/32/128) — это три разных
рассказа об одном и том же журнале.

## Блок 2 — Аудит, кривые, колено

Прежде чем верить любому числу из журнала — **аудит**: у доверенного журнала не бывает
пропавших терминалов, дублей ID, чужих меток волн и невозможных времён (терминал раньше
прихода). Ваша `audit_records` классифицирует повреждения и **никогда не чинит** их.

Дальше — три графика с аннотациями, и на каждом отмечено главное: на пропускной
способности — **колено** по правилу γ (последнее удвоение, окупившееся на ≥50%), на хвосте
задержек — **SLO 800 мс**, на исходах — доли в процентах. Читать их нужно зонами: подъём —
батчинг окупает чтение весов; плато — ресурсы заполнились, лишние клиенты покупают место
в очереди; обвал — бюджеты превращаются в таймауты. Среднее при этом может почти не
двигаться: хвост говорит раньше среднего, поэтому p50 и p95 всегда рядом.

И помните, как такие графики обманывают: выкинутые из знаменателя ошибки (плоская
«счастливая» кривая), показ только среднего (хвост невидим), молчаливые ретраи (отказы
возвращаются как успехи, спрос удваивается). У каждой ловушки — свой артефакт-защита в
этой лабе: доли исходов, p95 рядом с p50, запрет ретраев в драйвере.

<img src="https://levshaazz.github.io/llm-serving-mastery/Lectures/assets/figures/t6-scheduler-timeline-light.svg" width="860"/>

Часы шедулера (иллюстрация): зелёный столбец — префилл нового запроса забирает шаг у всех
резидентов; красный — преемпция, янтарный — пересчёт. «Задержка выросла, а ошибок нет» —
это оно. Диагностика по знакам: латентность без ошибок — интерференция префиллов;
остановка завершений — блокировка приёма; дублирование ранних токенов — преемпция с
пересчётом. Один и тот же график, три разных механизма, три разных лекарства.

In [ ]:
def audit_records(records: Sequence[RequestRecord], wave: int | None = None) -> AuditSummary:
    """Audit one wave's journal (or all waves when ``wave`` is None).

    Parameters
    ----------
    records : sequence of RequestRecord
        Raw journal records, possibly damaged.
    wave : int or None
        Restrict to this wave label; None audits everything.

    Returns
    -------
    AuditSummary
        records (considered), unique_ids, missing_terminal (first_byte or
        terminal absent), duplicate_ids, foreign_waves (record['wave'] != wave
        when restricted), impossible_time (terminal < arrival), complete.

    Notes
    -----
    A record counts as complete only when it is unique, in-wave, fully timed and
    its timestamps are ordered. Nothing is ever silently repaired or dropped.
    """
    raise NotImplementedError('TODO: classify each record; never repair it')

# --- самопроверка на битом журнале ---
def _rec(rid: str, wave: int = 8, arr: float = 1.0, fb: float | None = 1.1,
         term: float = 1.2) -> RequestRecord:
    """Build one synthetic journal record for the audit fixture.

    Parameters
    ----------
    rid : str
        Request identity.
    wave : int, default 8
        Wave label.
    arr : float, default 1.0
        Arrival timestamp, monotonic seconds.
    fb : float or None, default 1.1
        First-byte timestamp; None simulates a missing field.
    term : float, default 1.2
        Terminal timestamp.

    Returns
    -------
    RequestRecord
        A completed-style record with the given clocks.
    """
    return RequestRecord(request_id=rid, wave=wave, client_id=0, app_id='A', case_id='c',
                         input_tokens=100, max_tokens=32, arrival_ts=arr, first_byte_ts=fb,
                         terminal_ts=term, output_tokens=32, http_status=200,
                         outcome='completed')
_fx = [_rec('ok-1'), _rec('ok-2'), _rec('dup'), _rec('dup'), _rec('no-fb', fb=None),
       _rec('back', term=0.5), _rec('foreign', wave=999)]
assert audit_records(_fx, wave=8) == AuditSummary(records=6, unique_ids=5,
    missing_terminal=1, duplicate_ids=1, foreign_waves=1, impossible_time=1, complete=3)
print('audit_records: битый журнал классифицирован точно; ничего не «починено»')

In [ ]:
def throughput_by_wave(records: Sequence[RequestRecord]) -> dict[int, float]:
    """Output throughput per wave: completed output tokens / wave wall seconds.

    Parameters
    ----------
    records : sequence of RequestRecord
        Audited journal records across waves.

    Returns
    -------
    dict[int, float]
        wave -> tokens per second. The wall window runs from the earliest
        arrival to the latest terminal IN that wave; only records with outcome
        'completed' contribute tokens; failed requests still consume wall time.

    Raises
    ------
    ValueError
        If a wave contains fewer than two records or no completed record.
    """
    raise NotImplementedError('TODO: tokens of completed requests over the wave wall window')

# --- самопроверка ---
_tw = [_rec('a', wave=8, arr=0.0, fb=0.1, term=1.0), _rec('b', wave=8, arr=0.2, fb=0.3, term=2.0),
       _rec('c', wave=32, arr=0.0, fb=0.1, term=4.0), _rec('d', wave=32, arr=0.0, fb=0.1, term=1.0)]
_tp = throughput_by_wave(_tw)
assert _tp[8] == 64 / 2.0 and _tp[32] == 64 / 4.0, _tp
print('throughput_by_wave: окна и токены сходятся с ручным счётом')

In [ ]:
def ttft_percentiles(records: Sequence[RequestRecord], wave: int,
                   percentiles: tuple[int, ...] = (50, 95)) -> Percentiles:
    """Client-observed TTFT percentiles for one wave, completed requests only.

    Parameters
    ----------
    records : sequence of RequestRecord
        Audited journal.
    wave : int
        Wave label to restrict to.
    percentiles : tuple[int, ...], default (50, 95)
        Nearest-rank percentiles to report.

    Returns
    -------
    Percentiles
        wave, count (denominator: completed with first_byte), p50_ms, p95_ms.

    Raises
    ------
    ValueError
        If the wave has no completed request with a first-byte timestamp.
    """
    raise NotImplementedError('TODO: nearest-rank percentiles over completed TTFTs')

# --- самопроверка ---
_tt = [_rec(f't{i}', wave=8, arr=0.0, fb=0.010 * (i + 1), term=0.5) for i in range(100)]
_tpct = ttft_percentiles(_tt, 8)
assert _tpct['count'] == 100 and abs(_tpct['p50_ms'] - 500.0) < 1e-9 \
       and abs(_tpct['p95_ms'] - 950.0) < 1e-6, _tpct
print('ttft_percentiles: ближайший ранг p50/p95 сходится')

In [ ]:
def detect_knee(throughput: dict[int, float], min_gain: float) -> int:
    """Last wave whose throughput still gained >= min_gain over the previous wave.

    Parameters
    ----------
    throughput : dict[int, float]
        wave -> tokens/s, at least two waves, keys interpreted in sorted order.
    min_gain : float
        Required relative gain (0.5 means +50%%) to still count as growing.

    Returns
    -------
    int
        The knee wave: the largest wave whose gain over its predecessor is
        still >= min_gain. Waves before any qualifying gain return the first key.

    Raises
    ------
    ValueError
        If fewer than two waves or any value is not a positive finite number.
    """
    raise NotImplementedError('TODO: walk sorted waves and keep the last strong gain')

# --- самопроверка ---
assert detect_knee({8: 100.0, 32: 200.0, 64: 260.0, 128: 250.0}, 0.5) == 32
assert detect_knee({8: 100.0, 32: 200.0, 64: 260.0, 128: 250.0}, 0.2) == 64
print('detect_knee: пороги усиления ведут себя на синтетической кривой')

In [ ]:
tp = throughput_by_wave(all_records)
knee = detect_knee(tp, min_gain=0.5)
plot_curve(WAVES, [tp[w] for w in WAVES], 'выходные токены/с (completed)',
           'throughput.png', title=f'Пропускная способность; колено = {knee} (правило γ=0.5)',
           knee=knee)

In [ ]:
p95 = [ttft_percentiles(all_records, w)['p95_ms'] for w in WAVES]
plot_curve(WAVES, p95, 'TTFT p95, мс (completed)', 'ttft-p95.png',
           title='Хвост первого токена против SLO', slo_ms=800.0)

In [ ]:
outcome_shares(all_records, WAVES, 'outcome-shares.png')

In [ ]:
pred = json.load(open(os.path.join(RESULTS, 'predictions.json')))
print(f'измеренное колено = {knee}   (' +
      ', '.join(f'{w}: {tp[w]:.0f} ток/с' for w in WAVES) + ')' + chr(10) +
      f"ваш замороженный прогноз = {pred['predicted_knee_wave']}; "
      f"арифметический потолок = {pred['ceiling_requests']} запросов")

## Блок 3 — Решение под SLO (+ stretch)

Судья курса считает геометрическое среднее производительности на 32 и 64 клиентах и не
пускает дальше p95 TTFT > 0.8 с на 64. Ваше лабораторное SLO той же формы: **наибольшая
волна, где p95 ≤ 800 мс и доля ошибок ≤ 5%**, с названной причиной-ограничением. Ниже —
что показал измеренный прогон на настоящей T4 (18 октября 2026): кривая не рухнула —
прочитайте его вывод, прежде чем ожидать «обвал».

<img src="https://levshaazz.github.io/llm-serving-mastery/Lectures/assets/figures/t6-measured-saturation-light.svg" width="860"/>

**Мемо** — три обязательные строки: (1) выбранная волна против замороженного прогноза —
совпало или нет и почему; (2) какой ресурс связал первым и на каких графиках это видно
(p95 против водопада против долей исходов); (3) одно альтернативное объяснение, которое
этот прогон не исключает (например, разовый джиттер волны или прогрев). Вердикт
«неудобной» правды — например, «колено не найдено в диапазоне» — полноценный результат.

**Stretch (по желанию):** приложение B делит сервер; интерференция меряется по каждому
приложению отдельно, усреднять в один бар нельзя; защита — in-flight cap K ∈ {1,2,4,8,16};
девелопмент-пара → заморозка K с письменным прогнозом → одна held-out пара. K=0 запрещён.
**Вердикты accept / reject / inconclusive равноправны** — оценивается заморозка.

In [ ]:
def choose_operating_point(throughput: dict[int, float], ttft_p95_ms: dict[int, float],
                          error_rate: dict[int, float], ttft_slo_ms: float,
                          max_error_rate: float) -> Decision:
    """Largest wave whose p95 TTFT and error rate both satisfy the SLO.

    Parameters
    ----------
    throughput, ttft_p95_ms, error_rate : dict[int, float]
        Per-wave measured curves (all waves present in each).
    ttft_slo_ms : float
        Maximum allowed p95 TTFT in milliseconds.
    max_error_rate : float
        Maximum allowed share of non-completed records, in [0, 1].

    Returns
    -------
    Decision
        The chosen wave with its numbers and a one-sentence reason naming the
        binding constraint. When no wave qualifies, wave is -1 and the reason
        must say which constraint failed everywhere.

    Raises
    ------
    ValueError
        If the dicts disagree on wave labels or any input is out of range.
    """
    raise NotImplementedError('TODO: largest SLO-satisfying wave, binding constraint named')

# --- самопроверка ---
_d = choose_operating_point(throughput={8: 100.0, 32: 200.0, 64: 260.0},
                             ttft_p95_ms={8: 200.0, 32: 700.0, 64: 1500.0},
                             error_rate={8: 0.0, 32: 0.01, 64: 0.02},
                             ttft_slo_ms=800.0, max_error_rate=0.05)
assert _d['wave'] == 32 and 'TTFT' in _d['reason'], _d
print('choose_operating_point: наибольшая волна под SLO выбирается верно')

In [ ]:
# Все волны против SLO одним взглядом; выбранная строка подсвечена зелёным.
err = {w: 1 - audit_records(all_records, wave=w)['complete'] /
          max(1, audit_records(all_records, wave=w)['records']) for w in WAVES}
p95d = {w: ttft_percentiles(all_records, w)['p95_ms'] for w in WAVES}
decision = choose_operating_point(tp, p95d, err, ttft_slo_ms=800.0, max_error_rate=0.05)
tbl = summary_rows(all_records, WAVES)
tbl['p95 ≤ 800 мс'] = ['да' if p95d[w] <= 800 else 'НЕТ' for w in tbl.index]
tbl['ошибки ≤ 5%'] = ['да' if err[w] <= 0.05 else 'НЕТ' for w in tbl.index]
styled = tbl.style.apply(
    lambda row: ['background-color:#dcfce7'] * len(row)
    if row.name == decision['wave'] else [''] * len(row), axis=1)
display(styled.set_caption(
    f"выбранная волна: {decision['wave']} — {decision['reason']}"))

In [ ]:
memo_lines = [
    'TODO: выбранная волна против замороженного прогноза — совпало или нет и почему',
    'TODO: какой ресурс связал первым и на каких данных это видно',
    'TODO: одно альтернативное объяснение, которое этот прогон не исключает',
]
with open(os.path.join(RESULTS, 'decision.json'), 'w') as fh:
    json.dump(dict(decision=decision, memo=memo_lines, profile=PROFILE),
              fh, ensure_ascii=False, indent=1)
print('мемо сохранено в decision.json; заполните три TODO выше и сохраните тетрадь')

In [ ]:
class InFlightCapRunner:
    """Wrap a runner so at most ``k`` of THIS application's requests are in flight."""
    def __init__(self, runner: object, k: int) -> None:
        """Bind a runner and its in-flight budget.

        Parameters
        ----------
        runner : object
            Any runner with .request(case, max_tokens).
        k : int
            Cap, one of 1/2/4/8/16.

        Returns
        -------
        None
            Wrapper state; see request().
        """
        if k not in (1, 2, 4, 8, 16):
            raise ValueError('K must be one of 1/2/4/8/16')
        self._runner, self._sem = runner, threading.BoundedSemaphore(k)

    def request(self, case: dict, max_tokens: int) -> tuple[str, int]:
        """Forward one request while holding a cap slot.

        Parameters
        ----------
        case : dict
            Load case for the wrapped runner.
        max_tokens : int
            Output token cap.

        Returns
        -------
        tuple of (str, int)
            Whatever the wrapped runner returns.
        """
        with self._sem:
            return self._runner.request(case, max_tokens)

In [ ]:
def interference_delta(a_alone: Sequence[RequestRecord],
                      a_shared: Sequence[RequestRecord]) -> InterferenceReport:
    """Measure A's degradation when B shares the server.

    Parameters
    ----------
    a_alone, a_shared : sequence of RequestRecord
        A-only journal, and the MERGED A+B journal of the shared wave
        (app_id distinguishes records; the B count is read from it).

    Returns
    -------
    InterferenceReport
        a_throughput_ratio = shared / alone (>1 means A got FASTER — explain or
        suspect the run), a_ttft_p95_ratio = shared / alone, and completed counts.

    Raises
    ------
    ValueError
        If either journal has no completed A record for the wave.
    """
    raise NotImplementedError('TODO: ratios of shared to alone, completed counts')

# --- самопроверка ---
_ia = [_rec('a1', wave=32, arr=0.0, fb=0.2, term=2.0)]
_ib = [_rec('s1', wave=32, arr=0.0, fb=0.4, term=2.0),
       RequestRecord(request_id='b1', wave=32, client_id=1, app_id='B', case_id='x',
                     input_tokens=480, max_tokens=32, arrival_ts=0.0, first_byte_ts=0.5,
                     terminal_ts=2.0, output_tokens=32, http_status=200, outcome='completed')]
_ir = interference_delta(_ia, _ib)
assert abs(_ir['a_ttft_p95_ratio'] - 2.0) < 1e-9 and _ir['b_completed_shared'] == 1, _ir
print('interference_delta: отношения и счётчик B сходятся на фикстуре')

In [ ]:
if PROFILE == 'gpu':
    BASE, A6, B6, TMO6 = HttpRunner(), gpu_cases_A, gpu_cases_B, 60.0
else:
    BASE, A6, B6, TMO6 = MockQueueServer(), MOCK_CASES, MOCK_CASES, 12.0
a_alone = run_wave(BASE, 'A', A6, 32, per_client=3, timeout_s=TMO6)
s6_rows = []
for k in (1, 2, 4, 8, 16):
    shared = run_shared_wave(BASE, k, A6, B6, n_a=32, n_b=16, per_client=3, timeout_s=TMO6)
    rep = interference_delta(a_alone, shared)
    s6_rows.append(dict(K=k, thr_ratio=round(rep['a_throughput_ratio'], 2),
                        ttft_ratio=round(rep['a_ttft_p95_ratio'], 2),
                        B_done=rep['b_completed_shared']))
display(pd.DataFrame(s6_rows).set_index('K'))

In [ ]:
# Размен одной картинкой: пропускная способность A (столбцы) против хвоста TTFT (линия) по K.
fig, ax1 = plt.subplots(figsize=(6.5, 4))
ks = [row['K'] for row in s6_rows]
ax1.bar(ks, [row['thr_ratio'] for row in s6_rows], color='#2563eb', alpha=0.7,
        label='A: доля пропускной способности')
ax1.set_xlabel('in-flight cap K для B')
ax1.set_ylabel('A thr (shared / alone)')
ax1.set_ylim(0, 1.05)
ax2 = ax1.twinx()
ax2.plot(ks, [row['ttft_ratio'] for row in s6_rows], marker='o', color='#dc2626',
         label='A: p95 TTFT (shared / alone)')
ax2.set_ylabel('A p95 TTFT ratio', color='#dc2626')
ax1.set_title('Интерференция: что платит A за пропуск B')
fig.legend(loc='lower center', ncol=2, fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, 's6-tradeoff.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
frozen_k = 4   # TODO: ваш K из таблицы и графика выше
k_forecast = 'TODO: прогноз — что покажет held-out пара для A при этом K'
with open(os.path.join(RESULTS, 'policy.json'), 'w') as fh:
    json.dump(dict(k=frozen_k, forecast=k_forecast, frozen_at=time.time()),
              fh, ensure_ascii=False, indent=1)
print(f'K={frozen_k} заморожен — политика не меняется после этой точки')

In [ ]:
HA = heldout_A if PROFILE == 'gpu' else MOCK_CASES
HB = heldout_B if PROFILE == 'gpu' else MOCK_CASES
heldout = run_shared_wave(BASE, frozen_k, HA, HB, n_a=32, n_b=16, per_client=3, timeout_s=TMO6)
rep = interference_delta(a_alone, heldout)
print(f"held-out: A thr ratio = {rep['a_throughput_ratio']:.2f}, "
      f"A p95 TTFT ratio = {rep['a_ttft_p95_ratio']:.2f}, "
      f"B completed = {rep['b_completed_shared']}" + chr(10) +
      'вердикт — TODO: accept / reject / inconclusive одной фразой, привязанной к прогнозу')

In [ ]:
artifacts = {}
for name in ('predictions.json', 'raw-journal.jsonl', 'decision.json',
             'policy.json', 'throughput.png', 'ttft-p95.png', 'outcome-shares.png',
             'waterfall.png', 's6-tradeoff.png'):
    p = os.path.join(RESULTS, name)
    if os.path.exists(p):
        artifacts[name] = hashlib.sha256(Path_bytes(p)).hexdigest()
with open(os.path.join(RESULTS, 'manifest.json'), 'w') as fh:
    json.dump(dict(profile=PROFILE, artifacts=artifacts), fh, ensure_ascii=False, indent=1)
import shutil  # noqa: E402
shutil.make_archive('/content/t06-' + PROFILE + '-bundle', 'gztar', RESULTS)
print(f'{len(artifacts)} артефактов хэшировано; profile={PROFILE}; '
      f'bundle: /content/t06-{PROFILE}-bundle.tar.gz')

In [ ]:
# GPU PROFILE ONLY. Owned teardown of the service started in t06-gpu-bootstrap.
if PROFILE != 'gpu':
    print('skipped: gpu-only cell (profile is mock)')
else:
    server.terminate()
    try:
        server.wait(timeout=30)
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait()
    print('server exit code', server.returncode,
          '- delete the runtime only after the bundle is downloaded')

## Справочник: термины и числа лабы

Держите эту страницу под рукой при защите: каждый термин
здесь привязан к полю журнала или функции, которая его считает, а каждое число — к
происхождению (арифметика, объявление или измерение). Если на вопрос «откуда это
число?» вы можете показать строку этой таблицы — ответ защищаем.

| Термин | Что это | Где живёт |
|---|---|---|
| TTFT | время от прихода до первого токена: очередь + префилл | журнал, `first_byte_ts` |
| TPOT | время на выходной токен после первого | разность terminal и first_byte |
| wall-окно волны | от первого прихода до последнего терминала | знаменатель throughput |
| p50 / p95 | ближайший ранг; p95 — «каждый двадцатый» | `ttft_percentiles` |
| колено (γ=0.5) | последнее удвоение конкурентности, окупившееся на ≥50% | `detect_knee` |
| KV-цена позиции | 2×36×2×128×2 = 36 864 байта ≈ 36 КиБ | `kv_budget_bytes` |
| потолок 125 | (15360 − 6350) МиБ ÷ 72 МиБ на запрос | арифметика, не политика |
| SLO лабы | p95 ≤ 800 мс и ошибки ≤ 5% | `choose_operating_point` |
| S судьи | корень из произведения tok/s на 32 и 64 клиентах | лидерборд курса |

Числа mock-профиля — объявленная синтетика (формула в свёрнутой клетке драйвера); числа
gpu-профиля — измерение с провенансом. Смешивать их в одном выводе нельзя — это главное правило честности всей лабы и главный критерий на защите.


## Что вы теперь умеете — и чего эта лаба НЕ доказывает

**Умеете:** читать нагрузку глазами (тексты, длины, водопад); посчитать бюджет памяти до
измерений и свести прогноз с фактом; провести
контролируемые волны с журналом, где ошибки — данные; построить водопад и три
аннотированных графика; найти колено по объявленному правилу; выбрать и защитить рабочую
точку под SLO с названным ограничением; (stretch) измерить интерференцию и защитить
её капом с заморозкой до held-out.

**Чего этот прогон НЕ доказывает:** mock-модель — объявленная синтетика, её числа не
измерение vLLM; один прогон — не статистика, и точка на кривой может сдвинуться на
повторе; burst-волны по три запроса на клиента — не устоявшаяся нагрузка и не протокол
судьи (устоявшийся поток с ignore_eos); free-GPU-квота Colab не проверена;
learner-validation не проводилась. Измерение — только gpu-профиль с провенансом, и даже
оно — один прогон на одной машине.

**Куда дальше.** Тема 7 даёт ручки: `--max-num-seqs`, `--gpu-memory-utilization`,
`--max-model-len`, chunked prefill — те же механизмы, которые вы научились видеть, но
теперь как ручки: объявленный поиск, таблица конфигураций, замороженный производственный
профиль под тем же SLO. Навыки этой лабы — её фундамент: без честного журнала и чтения
хвостов любой тюнинг — настройка на удачу.